<a href="https://colab.research.google.com/github/alanapooler827/554Homework9/blob/main/ST554_HW9_Pooler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = SparkSession.builder.getOrCreate()

# Homework 9
In this homework, we will be fitting and comparing three different models on a dataset using spark MLlib.

## Dataset: Sleep Health

We will be the sleep health data set from Kaggle. It contains 100,000 synthetic patient records and includes demographic, lifestyle, sleep information, and conginitive scores. We will attempt to sleep disorder risk using the information provided in the data set.

This data is synthetic to protect patient privacy, but all distributions, correlations, and occupation-specific profiles are calibrated against peer-reviewed sources.

[View the data set on Kaggle.](https://www.kaggle.com/datasets/mohankrishnathalla/sleep-health-and-daily-performance-dataset)

To get started, let's read in the data set.

In [49]:
sleep_data = pd.read_csv('sleep_health_dataset.csv')

# convert from pandas df to spark df
df = spark.createDataFrame(sleep_data)

# view first 5 rows
df.show(3)

+---------+---+------+-----------------+----+-------+------------------+-------------------+--------------+---------------------+------------------+-----------------------+----------------------+------------------------+---------------------------+------------+--------------+-----------------+------------+-------------------+----------+-----------------------+----------------------+--------------+----------+------------------------+----------------------+------+--------+---------------------------+-------------------+-----------+
|person_id|age|gender|       occupation| bmi|country|sleep_duration_hrs|sleep_quality_score|rem_percentage|deep_sleep_percentage|sleep_latency_mins|wake_episodes_per_night|caffeine_mg_before_bed|alcohol_units_before_bed|screen_time_before_bed_mins|exercise_day|steps_that_day|nap_duration_mins|stress_score|work_hours_that_day|chronotype|mental_health_condition|heart_rate_resting_bpm|sleep_aid_used|shift_work|room_temperature_celsius|weekend_sleep_diff_hrs|seaso

## Data Preparation and Transformations

First, we need to split our data into training and testing sets.

We will use 80% of the data for the training set, leaving 20% for the testing set.

In [50]:
train, test = df.randomSplit([0.8, 0.2], seed = 42)
print(train.count(), test.count())

80062 19938


The sleep disorder risk variable has four categories- healthy, mild, moderate, and severe. It is very imbalanced, with the majority of subjects in the 'healthy' category, and the least amount of subjects in the 'severe' category.

To turn this sleep disorder risk into a binary variable and make it more balanced, let's reclassify the categories to 'healthy' and 'at risk' using SQLTransformer(). For the purpose of modeling, we will encode healthy as 0 and at risk as 1.

In [51]:
label_trans = SQLTransformer(
    statement="""
        select *,
            case
                when lower(sleep_disorder_risk) = 'healthy' then 0.0
                else 1.0
            end as label
        from __THIS__
    """
)

We will also use SQLTransformer() to create an interaction term to use in one of our models. We will multiply sleep duration by stress score to create the interaction term.

In [ ]:
interaction_trans = SQLTransformer(
    statement="""
        SELECT
            *,
            sleep_duration_hrs * stress_score AS interaction
        FROM __THIS__
    """
)

Next, we will convert the day_type variable to binary. It currently has the values 'Weekday' and 'Weekend', so we will use StringIndexer() to convert the values to 0 and 1.

In [56]:
day_indexer = StringIndexer(
    inputCol='day_type',
    outputCol='day_idx',
    handleInvalid='keep'
)

We will use VectorAssembler() to put all of the predictors into one column called 'features'. We will create one assembler without the interaction term and one with it.

In [73]:
assembler = VectorAssembler(
    inputCols=[
        'age',
        'exercise_day',
        'sleep_aid_used',
        'sleep_duration_hrs',
        'stress_score',
        'day_idx',
        'rem_percentage',
        'deep_sleep_percentage',
        'cognitive_performance_score'
    ],
    outputCol='features_raw'
)

# assembler with interaction term
assembler_inter = VectorAssembler(
    inputCols=[
        'age',
        'exercise_day',
        'sleep_aid_used',
        'sleep_duration_hrs',
        'stress_score',
        'day_idx',
        'rem_percentage',
        'deep_sleep_percentage',
        'cognitive_performance_score',
        'interaction'
    ],
    outputCol='features_raw'
)

In homeworks 7 and 8, we saw a big improvement in the classification models after scaling the features, so we will use StandardScalar() to do that here as well.

In [67]:
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withMean=False,
    withStd=True
)

We will be using AUC (area under the ROC curve) as the evaluation metric for our three models. This metric tells us how well the model distinguishes between two classes- healthy and at risk in this case. AUC can take on values between 0.5 and 1. An AUC of 0.5 means that the model is no better than random guessing, and an AUC of 1 means that the model perfectly classifies the responses.

In [54]:
evaluator = BinaryClassificationEvaluator(
    labelCol='label',
    metricName='areaUnderROC'
)

## Model 1: Logistic Regression

Now we can build our first model, which will be a logistic regression model. A logistic regression model is a supervised machine learning model used when the outcome is binary (has two options). The model predicts the probability that a specific outcome occurs based on the input data it is given.

First we will use Pipeline() to apply the transformation to the training and test sets.

Then we will use CrossValidator() to test different values of the regularization parameter.

Lastly, we will evaluate the model on the test set using our evaluation metric, AUC.

In [72]:
# define logistic regression model
lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100)

# define pipeline
lr_pipeline = Pipeline(
    stages=[
        label_trans,
        day_indexer,
        assembler,
        scaler,
        lr
    ]
)

# define parameter grid
lr_param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.1])
    .build()
)

# run cross validation
lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator,
    numFolds=3
)

# fit model to training set
lr_model = lr_cv.fit(train)

# make predictions on test set
lr_preds = lr_model.transform(test)

# calculate AUC
lr_auc = evaluator.evaluate(lr_preds)
print('Logistic Regression AUC:', lr_auc)

Logistic Regression AUC: 0.903204304579424
